# GDELT (BigQuery) → extraction locale (raw)

Ce notebook exécute une requête BigQuery sur la table publique GDELT et sauvegarde le résultat dans `data/raw/`.

## Auth BigQuery (dans l'ordre)
- `GOOGLE_APPLICATION_CREDENTIALS` (chemin vers un JSON Service Account)
- `credentials/credentials.json` à la racine du repo
- sinon ADC (Application Default Credentials)

In [4]:
from __future__ import annotations
import logging
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

import scripts.data_pipeline as dp

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s - %(message)s")
sys.path.append(str(Path().resolve().parent))

PROJECT_ROOT = Path().resolve().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [5]:
import importlib
importlib.reload(dp)

<module 'scripts.data_pipeline' from 'D:\\Documents\\Hackathon_iSHEEROXDatacamp\\scripts\\data_pipeline.py'>

In [8]:
query = """
SELECT *
FROM `gdelt-bq.gdeltv2.events`
WHERE ActionGeo_CountryCode = 'BN'
  AND (SQLDATE BETWEEN 20250101 AND 20251231)
""".strip()

client = dp.get_bq_client()
df = dp.extract_raw_data(client, query)

raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
dp.load_raw_data(df, raw_dir / "gdelt_bn_2025.csv")
df.head()

2026-04-30 20:29:34,554 INFO scripts.data_pipeline - Using service account credentials from D:\Documents\Hackathon_iSHEEROXDatacamp\credentials\credentials.json (project=warm-abacus-494720-h0)
2026-04-30 20:29:34,557 INFO scripts.data_pipeline - Running query (120 chars)
2026-04-30 20:30:00,165 INFO scripts.data_pipeline - DataFrame saved to D:\Documents\Hackathon_iSHEEROXDatacamp\data\raw\gdelt_bn_2025.csv


,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_ADM2Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
0,1292975459,20250307,202503,2025,2025.1836,NaN,NaN,NaN,NaN,NaN,...,5,"Borgou, Borgou, Benin",BN,BN10,5886,9.75,2.75,-1332814,20260307140000,http://www.nigeriasun.com/news/278907556/benin...
1,1289290869,20250215,202502,2025,2025.1233,AFRCVL,AFRICA,AFR,NaN,NaN,...,1,Benin,BN,BN,NaN,9.50,2.25,BN,20260215143000,https://www.newsghana.com.gh/african-leaders-c...
2,1285645948,20250124,202501,2025,2025.0658,HLH,HOSPITAL,NaN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.50,2.25,BN,20260124103000,https://punchng.com/senator-decries-poor-state...
3,1285645959,20250124,202501,2025,2025.0658,LEG,LAWMAKER,NaN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.50,2.25,BN,20260124103000,https://punchng.com/senator-decries-poor-state...
4,1285645960,20250124,202501,2025,2025.0658,LEG,SENATE,NaN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.50,2.25,BN,20260124103000,https://punchng.com/senator-decries-poor-state...


In [9]:
job = client.query(query)

print(job.state)
print(job.error_result)

DONE
None
